# Chapter 5 — IAAIS Uncertainty Module

This notebook demonstrates exact categorical Bayesian updating, expected utility, particle filtering, and empirical calibration. Probabilities remain separate from deterministic Knowledge Base facts.


In [ ]:
import math
from iaais.uncertainty import (
    ParticleFilter,
    UncertaintyModule,
    bayesian_update,
)


## 1. Exact Bayesian update

For each possible activity state, provide the likelihood of the observed sensor evidence. Bayes' theorem combines those likelihoods with the prior.


In [ ]:
prior = {"squat": 0.60, "lunge": 0.40}
likelihoods = {"squat": 0.90, "lunge": 0.20}  # P(sensor vote | state)
posterior, evidence_probability = bayesian_update(prior, likelihoods)
print("P(sensor evidence):", round(evidence_probability, 3))
print("Posterior:", dict(posterior.probabilities))


## 2. Keep the classifier's full output

A top class with confidence 0.73 is not enough to reconstruct a multiclass distribution. Store the full output from the classifier.


In [ ]:
uncertainty = UncertaintyModule()
report = uncertainty.observe_classifier_prediction(
    "activity",
    {"squat": 0.73, "lunge": 0.19, "rest": 0.08},
    metadata={"model_version": "demo"},
)
print(dict(report.posterior.probabilities))
print("Most likely:", report.most_likely, report.most_likely_probability)


## 3. Pass expected utility to planning

The planner can compare actions using expected utility rather than assuming one outcome is certain.


In [ ]:
planning_uncertainty = UncertaintyModule()
planning_uncertainty.set_prior("activity", {"squat": 0.60, "lunge": 0.40})
belief = planning_uncertainty.update_categorical(
    "activity", "side sensor favors squat", likelihoods, source="side_sensor"
)
utility = {"squat": 10.0, "lunge": -2.0}
print("Expected utility:", round(planning_uncertainty.expected_utility("activity", utility), 3))


## 4. Update a continuous sensor belief

A particle filter represents uncertainty over a noisy sensor value. The likelihood below favors particle states near the measurement.


In [ ]:
particles = ParticleFilter(
    particles=[0.0, 1.0, 2.0, 3.0, 4.0],
    resample_threshold=0.5,
    seed=7,
)
sensor_report = particles.update(
    observation=2.4,
    likelihood=lambda state, reading: math.exp(-0.5 * ((state - reading) / 0.8) ** 2),
    source="accelerometer",
)
print("Posterior mean:", round(sensor_report.mean, 3))
print("Posterior variance:", round(sensor_report.variance, 3))
print("Effective sample size:", round(sensor_report.effective_sample_size, 2))


## 5. Measure calibration when labels arrive

A probability is not empirically calibrated just because it is normalized. Record predictions with their eventual labels to calculate calibration metrics.


In [ ]:
for actual in ("squat", "squat", "squat", "lunge"):
    calibration = uncertainty.record_calibration_case(
        "activity",
        {"squat": 0.80, "lunge": 0.20},
        actual,
        bin_count=5,
    )
print("Labeled cases:", calibration.sample_count)
print("Brier score:", round(calibration.brier_score, 3))
print("Expected calibration error:", round(calibration.expected_calibration_error, 3))


## Interpretation

The posterior and evidence history are decision support. They do not diagnose injuries, certify technique, or automatically prescribe treatment. Confirmed Knowledge Base facts can condition beliefs; proposed or conflicted facts require review before they can be treated as deterministic.
